# 최종과제: 쇼핑몰 고객 문의 처리 에이전트

쇼핑몰 고객 문의를 처리하는 에이전트를 실행합니다.

## 과제 목표

아래 항목을 확인합니다.

- 주문 정보를 Tool로 조회할 수 있는가
- 요청에 포함된 민감정보가 모델 입력과 Tool 인자에서 보호되는가
- 고객 문의 접수가 승인 후 실행되는가
- 같은 `thread_id`로 후속 질문을 했을 때 이전 맥락이 유지되는가

## 과제 시나리오

고객이 **배송 지연 문의**를 남기면서 **결제 카드번호**까지 함께 적었다고 가정합니다.

> 주문 **ORD-1001** 배송이 아직 도착하지 않았습니다.  
> 결제 카드번호는 **5105-1051-0510-5100** 입니다.  
> **우선순위 높음**으로 고객 문의를 접수해주세요.

에이전트는 **주문 정보**를 조회한 뒤, **고객 문의 접수**를 시도합니다.  
이때 **카드번호 원문**이 그대로 남아 있으면 안 됩니다.

고객 문의 접수는 실제 업무로 이어질 수 있는 작업이므로 바로 실행되지 않고 **승인 대기 상태**에서 멈춥니다.  
PII 처리가 정상적으로 되었는지 확인한 뒤 **승인 후 재개**합니다.

## 전체 흐름

요청은 PII 미들웨어를 거쳐 에이전트에 들어갑니다.  
그 뒤 조회 Tool과 승인 대기 Tool을 거쳐 결과가 만들어집니다.

<img src="image/final_assignment_support_agent_flow.svg" width="820">

## 제출 형태

제출물은 **캡처 보고서 1개**입니다.

- 파일명: `2.수행평가 제출양식_본인이름.docx` 
    - 본인 이름으로 변경해서 제출. 
    - 예시) 2.수행평가 제출양식_김민수.docx
- 보고서 분량: 1~3쪽
- 실행한 노트북 파일은 선택 제출입니다.

보고서에는 아래 실행 결과를 넣습니다.

| 번호 | 필수 캡처 | 위치 |
|---|---|---|
| 1 | 주문 조회 Tool 실행 결과 | `3. 주문 조회 Tool 확인` |
| 2 | PII 처리 확인과 승인 대기 정보 | `4. PII 처리와 고객 문의 접수 요청` |
| 3 | 승인 후 재개 결과 | `5. 승인 후 재개` |
| 4 | 같은 `thread_id` 후속 질문 결과 | `6. 같은 thread_id로 후속 질문` |

이 노트북의 카드번호는 테스트용 값입니다.

## 채점 기준

총점은 100점입니다. 수료 기준은 60점입니다.

| 항목 | 점수 | 채점 기준 |
|---|---:|---|
| 제출 형식 | 20점 | 지정된 파일명 형식의 DOCX 보고서 제출 |
| 주문 조회 Tool 확인 | 20점 | `lookup_order` 실행 결과 포함 |
| PII 처리 확인 | 20점 | 카드번호 원문이 입력 기록이나 Tool 인자에 그대로 남지 않음 |
| 승인 후 재개 확인 | 20점 | 승인 대기 정보와 승인 후 고객 문의 접수 결과 포함 |
| 후속 질문 및 설명 작성 | 20점 | 같은 `thread_id` 후속 질문 결과와 핵심 설명 포함 |

## 제출 전 확인

보고서에 아래 내용이 들어 있는지 확인하세요.

- 필수 캡처 항목
- Tool이 어떤 순서로 사용되었는지
- PII 미들웨어가 어떤 값을 처리했는지
- 고객 문의 접수에 승인이 필요한 이유
- 같은 `thread_id`를 사용한 이유

## 1. 환경 준비

`OPENAI_API_KEY`가 `.env`에 들어 있어야 합니다.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(dotenv_path=".env", override=True)
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY를 .env에 설정하세요."

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

print("환경 준비 완료")

## 2. Tool과 에이전트 준비

아래 코드의 `None`을 채우세요.

- PII 미들웨어: 카드번호를 감지하고 마스킹합니다.
- 승인 미들웨어: 고객 문의 접수 Tool 실행 전에 멈춥니다.
- 승인 결정: 중단된 실행을 승인 후 재개합니다.

`thread_id`는 승인 후 재개와 후속 질문을 같은 흐름으로 이어주는 값입니다.  
이 값은 바꾸지 않아도 됩니다.

실제 카드번호는 사용하지 마세요.

In [ ]:
# Tool, PII 미들웨어, 승인 미들웨어, 메모리를 준비합니다.
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware, PIIMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

In [ ]:
# 주문 조회 Tool이 사용할 예시 데이터입니다.
ORDER_DB = {
    "ORD-1001": {
        "customer": "김하나",
        "status": "배송사 인계 후 지연 확인",
        "last_update": "2026-06-10",
        "owner": "배송 클레임팀",
    },
    "ORD-1002": {
        "customer": "이준호",
        "status": "교환 접수",
        "last_update": "2026-06-09",
        "owner": "교환/반품팀",
    },
}

In [ ]:
# 주문번호로 현재 상태와 담당팀을 조회하는 Tool입니다.
@tool
def lookup_order(order_id: str) -> str:
    """주문번호로 고객명, 현재 상태, 최근 처리일, 담당팀을 조회합니다."""
    order_id = order_id.upper()
    info = ORDER_DB.get(order_id)
    if not info:
        return f"{order_id} 주문 정보를 찾지 못했습니다."
    return (
        f"주문={order_id}, 고객={info['customer']}, 상태={info['status']}, "
        f"최근처리일={info['last_update']}, 담당팀={info['owner']}"
    )


In [ ]:
# 고객 문의를 접수하는 Tool입니다. 실제 업무로 이어질 수 있어 승인 대상입니다.
@tool
def create_support_request(order_id: str, issue: str, priority: str) -> str:
    """고객 문의를 접수합니다. 고객 응대나 보상 검토로 이어질 수 있으므로 승인 후 실행해야 합니다."""
    return f"고객 문의 접수 완료: 주문={order_id}, 우선순위={priority}, 문의={issue}"


In [ ]:
# 조회 Tool과 고객 문의 접수 Tool을 에이전트에 연결합니다.
support_agent = create_agent(
    model=model,
    tools=[lookup_order, create_support_request],
    middleware=[
        PIIMiddleware(None, strategy=None),
        HumanInTheLoopMiddleware(interrupt_on={None: True}),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "너는 쇼핑몰 고객 문의 처리 에이전트입니다. "
        "주문 관련 요청이 들어오면 먼저 lookup_order로 주문 상태를 확인하세요. "
        "고객 문의 접수가 필요하면 create_support_request 도구를 사용하세요. "
        "사용자 메시지에 카드번호가 있더라도 원문 민감정보를 응답이나 Tool 인자에 그대로 쓰지 마세요. "
        "주문번호, 문의 내용, 우선순위가 요청에 있으면 추가 질문하지 마세요."
    ),
)

In [ ]:
# 같은 thread_id를 쓰면 승인 후 재개와 후속 질문이 같은 흐름으로 이어집니다.
support_config = {"configurable": {"thread_id": "final-support-pii-answer"}}

# 테스트용 카드번호입니다. 실제 카드번호를 넣지 마세요.
TEST_CARD_NUMBER = "5105-1051-0510-5100"

print("Tool과 에이전트 준비 완료")

## 3. 주문 조회 Tool 확인

먼저 Tool이 어떤 값을 반환하는지 직접 확인합니다.  
이 출력은 채점 기준의 “주문 조회 Tool” 증거가 됩니다.




In [ ]:
lookup_result = lookup_order.invoke({"order_id": "ORD-1001"})
print(lookup_result)

## 4. PII 처리와 고객 문의 접수 요청

아래 요청은 고객 문의 접수로 이어지는 작업입니다.  
카드번호가 포함되어 있으므로, 승인 전에 PII 처리가 되었는지 확인합니다.

PII 처리가 정상이라면 원문 카드번호가 입력 기록이나 Tool 인자에 그대로 남지 않아야 합니다.

In [ ]:
support_request = {
    "messages": [
        {
            "role": "user",
            "content": (
                f"주문 ORD-1001 배송이 아직 도착하지 않았습니다. "
                f"결제 카드번호는 {TEST_CARD_NUMBER} 입니다. "
                "우선순위 높음으로 고객 문의를 접수해주세요."
            ),
        }
    ]
}

paused_result = support_agent.invoke(support_request, support_config)
protected_input = paused_result["messages"][0].content
interrupts = paused_result["__interrupt__"]
first_action = interrupts[0].value["action_requests"][0]

print("[PII 처리된 사용자 입력]")
print(protected_input)

print("\n[승인 대기 정보]")
print(interrupts)

pii_check_passed = (
    TEST_CARD_NUMBER not in protected_input
    and TEST_CARD_NUMBER not in str(first_action["args"])
)

print("\n[PII 처리 확인]")
print(f"원문 민감정보가 입력/Tool 인자에 남아 있지 않음: {pii_check_passed}")

print("\n[승인 전 Tool 호출 초안]")
print({"name": first_action["name"], "args": first_action["args"]})

## 5. 승인 후 재개

아래 셀에서는 승인 결정을 채워 중단된 실행을 이어갑니다.

In [ ]:
review_decision = {"type": None}

print("[승인 결정]")
print(review_decision)

resumed_result = support_agent.invoke(
    Command(resume={"decisions": [review_decision]}),
    support_config,
)

print("\n[승인 후 결과]")
for message in resumed_result["messages"][-4:]:
    content = message.content or "(내용 없음)"
    print(f"{message.__class__.__name__}: {content}")

## 6. 같은 thread_id로 후속 질문

같은 `thread_id`를 사용하면 앞에서 진행한 요청 흐름과 이어서 대화할 수 있습니다.  
아래 질문의 답변을 확인하세요.




In [ ]:
follow_up_result = support_agent.invoke(
    {"messages": [{"role": "user", "content": "방금 요청한 주문의 담당팀과 최근 처리일을 다시 알려주세요."}]},
    support_config,
)

print("[같은 thread_id 후속 질문]")
print(follow_up_result["messages"][-1].content)

## 7. 캡처용 제출 전 확인

아래 셀은 제출 보고서에 넣을 실행 결과를 한곳에 모아 보여줍니다.  
실행 후 출력 영역을 캡처해서 보고서에 붙여도 됩니다.

In [ ]:
print("[캡처 1] 주문 조회 Tool 실행 결과")
print(lookup_result)

print("\n[캡처 2] PII 처리 확인과 승인 대기 정보")
print("[PII 처리된 입력]")
print(protected_input)
print("\n[PII 처리 통과 여부]")
print(pii_check_passed)
print("\n[승인 대기 정보]")
print(interrupts)

print("\n[캡처 3] 승인 후 고객 문의 접수 결과")
for message in resumed_result["messages"][-4:]:
    content = message.content or "(내용 없음)"
    print(f"{message.__class__.__name__}: {content}")

print("\n[캡처 4] 같은 thread_id 후속 질문 결과")
print(follow_up_result["messages"][-1].content)

## 8. 보고서에 넣을 설명 작성

제출 보고서 마지막에 아래 항목을 3~5줄로 작성하세요. 채점자는 이 부분을 보고 설명 점수를 부여합니다.

- `PIIMiddleware`가 처리한 민감정보:
- `lookup_order`가 호출되는 상황:
- `create_support_request`에 승인 절차가 필요한 이유:
- 같은 `thread_id`로 후속 질문을 보낸 이유:

작성 예시:

```text
요청에는 배송 지연 문의와 테스트용 카드번호가 포함되어 있다.
PIIMiddleware는 카드번호를 마스킹해 원문 민감정보가 Tool 인자에 남지 않게 했다.
lookup_order는 주문번호를 기준으로 고객명, 상태, 담당팀을 확인할 때 사용된다.
create_support_request는 실제 고객 응대나 보상 검토로 이어질 수 있으므로 사람의 승인을 거친다.
같은 thread_id를 사용하면 앞에서 요청한 주문 맥락을 이어서 확인할 수 있다.
```